In [ ]:
import pandas as pd
from hierarchicalforecast.utils import aggregate
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

In [ ]:
# ---------------------------------------------------------------
# Load from existing saved files (NOT recreating — reading what
# already exists on disk, per Pool A's actual environment pattern)
# ---------------------------------------------------------------
Y_product = pd.read_csv("../eabl parsed csv/Y_product.csv")
Y_product["ds"] = pd.to_datetime(Y_product["ds"])
S_product = pd.read_csv("../eabl parsed csv/S_product.csv", index_col=0)

# Raw inspection first, before any checks
print(Y_product.head())
print("Y_product shape:", Y_product.shape)


In [ ]:
print(S_product.head())
print("S_product shape:", S_product.shape)

In [ ]:

# Overwrite the old file with the corrected version
S_product.to_csv("../eabl parsed csv/S_product.csv")

print(S_product)

In [ ]:
results = {}  # collects True/False per check for the final summary

# ---------------------------------------------------------------
# 1 & 2. Column names check
# ---------------------------------------------------------------
expected_cols = ["unique_id", "ds", "y"]
actual_cols = list(Y_product.columns)
actual_cols


In [ ]:
results["columns_match"] = set(expected_cols).issubset(actual_cols)
results["columns_match"]

In [ ]:
print("Y_product columns:", actual_cols, "-> expected at least:", expected_cols)

In [ ]:
# ---------------------------------------------------------------
# 3. Dtype checks: ds must be datetime, y must be numeric
# ---------------------------------------------------------------
results["ds_is_datetime"] = pd.api.types.is_datetime64_any_dtype(Y_product["ds"])
results["ds_is_datetime"]

In [ ]:
results["y_is_numeric"] = pd.api.types.is_numeric_dtype(Y_product["y"])
results["y_is_numeric"]

In [ ]:

print("ds dtype:", Y_product["ds"].dtype, "| y dtype:", Y_product["y"].dtype)

In [ ]:
# ---------------------------------------------------------------
# 4. Every ds value must fall on a Monday (weekday() == 0)
# ---------------------------------------------------------------
non_monday = Y_product.loc[Y_product["ds"].dt.weekday != 0]
non_monday

In [ ]:
results["all_dates_are_monday"] = len(non_monday) == 0
results["all_dates_are_monday"]

In [ ]:
print("Non-Monday rows found:", len(non_monday))

In [ ]:
# ---------------------------------------------------------------
# 5. No duplicate (unique_id, ds) pairs
# ---------------------------------------------------------------
dupe_count = Y_product.duplicated(subset=["unique_id", "ds"]).sum()
dupe_count

In [ ]:
results["no_duplicate_pairs"] = dupe_count == 0
results["no_duplicate_pairs"]

In [ ]:
print("Duplicate (unique_id, ds) pairs:", dupe_count)
# Pool A's variant used: assert dupe_count == 0, "Duplicate rows found..."
# Equivalent check, fail-fast style — see merge log for why only the
# summary-dict version is kept as the primary form here.

In [ ]:

# ---------------------------------------------------------------
# 6. Inspect S_product's structure before assuming anything about it
# ---------------------------------------------------------------
print("\nS_product type:", type(S_product))


In [ ]:
print("S_product index sample:", list(S_product.index[:5]))


In [ ]:
print("S_product columns sample:", list(S_product.columns[:5]))

In [ ]:
S_product.columns

In [ ]:
S_product.columns

In [ ]:
if "unique_id" in S_product.columns:
    bottom_ids_from_S = set(S_product["unique_id"])
    print(f"The columns are {bottom_ids_from_S }\n")
    print(f"The length is {len(bottom_ids_from_S)}")
else:
    bottom_ids_from_S = set(S_product.columns)
    



In [ ]:
n_bottom = len(bottom_ids_from_S)
n_bottom

In [ ]:
results["bottom_level_count_is_222"] = n_bottom == 222
results["bottom_level_count_is_222"] 


In [ ]:
print("Bottom-level count implied by S_product:", n_bottom, "(expected 222)")

In [ ]:
print(S_product.shape)

In [ ]:
print(S_product.head())

In [ ]:
print(S_product.columns.tolist()[:10])


In [ ]:

# ---------------------------------------------------------------
# 7 & 8. Y_product vs S_product alignment
# ---------------------------------------------------------------
y_product_ids = set(Y_product["unique_id"].unique())
print(len(y_product_ids))

In [ ]:
print("\nTotal distinct unique_ids in Y_product (all hierarchy levels):", len(y_product_ids))
print("Distinct series in Y_product (nunique):", Y_product["unique_id"].nunique())

In [ ]:
print(len(bottom_ids_from_S))

In [ ]:
print(len(y_product_ids))

In [ ]:
missing_from_Y = bottom_ids_from_S - y_product_ids
print(missing_from_Y)

In [ ]:
results["Y_product_matches_S_bottom"] = len(missing_from_Y) == 0
print("Bottom-level ids missing from Y_product:", len(missing_from_Y))

In [ ]:
print(S_product.shape)
print(S_product.columns.tolist()[:10])
print(S_product.head())


In [ ]:
# ---------------------------------------------------------------
# 9a. Full per-series gap report — all series, not just one
# ---------------------------------------------------------------
gap_report = []
for uid, group in Y_product.groupby("unique_id"):
    start, end = group["ds"].min(), group["ds"].max()
    expected_weeks = pd.date_range(start, end, freq="W-MON")
    observed_weeks = set(group["ds"])
    missing_weeks = len(set(expected_weeks) - observed_weeks)
    zero_weeks = (group["y"] == 0).sum()
    gap_report.append({
        "unique_id": uid,
        "n_expected_weeks": len(expected_weeks),
        "n_observed_rows": len(group),
        "missing_weeks": missing_weeks,
        "zero_sale_weeks": zero_weeks,
    })


## Why this is useful

1. **It flags the worst-covered products.** Talisker, Lagavulin, Smirnoff Vodka Flavours, and Zacapa all show up at the top, meaning they have the least reliable, most incomplete histories. If you try to forecast these with a "normal" model (like `AutoETS`, which expects a fairly continuous series), it'll likely perform terribly, because there's barely any data to learn a pattern from.

2. **It directly feeds the intermittency classification work** (the ADI/CV² smooth / erratic / intermittent / lumpy routing built earlier). A product like Talisker with 55 missing weeks out of 57 is a textbook candidate for the "intermittent/lumpy" bucket, meaning it should go through `CrostonClassic` / `TSB` instead of `SeasonalNaive` / `AutoETS`.

3. **It's a business-side flag too.** A product appearing in only 2 of 57 weeks might be:
   - discontinued,
   - a very low-volume reserve item (Talisker and Lagavulin are both premium/reserve whisky, so small-volume, special-order items would naturally look like this), or
   - a data-collection issue (maybe that route/customer just doesn't report consistently).

   Worth checking with the business before assuming it's purely a forecasting problem.

> **In the simplest sentence:** this report tells you which products have the thinnest, most unreliable sales histories. Talisker `679073` only has real data for 2 out of 57 possible weeks, and it confirms that in this data, "no sales" always shows up as a **missing row**, never as an explicit zero.

In [ ]:
gap_report = pd.DataFrame(gap_report)

total_missing_weeks = gap_report["missing_weeks"].sum()
total_zero_weeks = gap_report["zero_sale_weeks"].sum()
print(f"\nTotal true gap-weeks across all series (NOT filled): {total_missing_weeks}")
print(f"Total recorded zero-sale weeks across all series: {total_zero_weeks}")
print(gap_report.sort_values("missing_weeks", ascending=False).head(10))


In [ ]:


# ---------------------------------------------------------------
# Missing-data heatmap — worst N products, gap vs. zero-sale vs. real sale
# ---------------------------------------------------------------
N = 30  # how many of the worst-covered products to show (222 would be unreadable)
worst_products = gap_report.sort_values("missing_weeks", ascending=False).head(N)["unique_id"]

full_calendar = pd.date_range(Y_product["ds"].min(), Y_product["ds"].max(), freq="W-MON")

status_matrix = pd.DataFrame(index=worst_products, columns=full_calendar, dtype=float)
for uid in worst_products:
    series = Y_product[Y_product["unique_id"] == uid].set_index("ds")["y"]
    reindexed = series.reindex(full_calendar)
    status_matrix.loc[uid] = reindexed.apply(lambda v: 0 if pd.isna(v) else (1 if v == 0 else 2)).values

cmap = mcolors.ListedColormap(["#d73027", "#fee08b", "#1a9850"])  # red / yellow / green
norm = mcolors.BoundaryNorm([-0.5, 0.5, 1.5, 2.5], cmap.N)

plt.figure(figsize=(14, 10))
plt.imshow(status_matrix.values.astype(float), aspect="auto", cmap=cmap, norm=norm)
plt.yticks(range(len(worst_products)), worst_products, fontsize=7)
plt.xticks([])
plt.xlabel("Weeks (full calendar range) →")
plt.title(f"Missing-Data Heatmap — Top {N} Worst-Covered Products")

legend_elements = [
    Patch(facecolor="#d73027", label="True gap (no row at all)"),
    Patch(facecolor="#fee08b", label="Recorded zero sale"),
    Patch(facecolor="#1a9850", label="Real sale recorded"),
]
plt.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, -0.05), ncol=3)
plt.tight_layout()
plt.show()


In [ ]:
Y_product["ds"].min()

In [ ]:
# ---------------------------------------------------------------
# 9b. Quick human-readable spot-check on ONE series
# (kept from Pool A's variant — simpler to read than the full table)
# ---------------------------------------------------------------
full_calendar = pd.date_range(Y_product["ds"].min(), Y_product["ds"].max(), freq="W-MON")
print(full_calendar)
sample_id = Y_product["unique_id"].iloc[0]
sample_series = (
    Y_product[Y_product["unique_id"] == sample_id]
    .set_index("ds")["y"]
    .reindex(full_calendar)
)


In [ ]:
full_calendar = pd.date_range(Y_product["ds"].min(), Y_product["ds"].max(), freq="W-MON")
print("Dataset start:", Y_product["ds"].min())
print("Dataset end:", Y_product["ds"].max())
print("Total weeks in full calendar (including gaps):", len(full_calendar))
print(results)


In [ ]:
print(f"\nSample series: {sample_id}")
print("True gaps (no row at all):", sample_series.isna().sum())
print("Recorded zeros:", (sample_series == 0).sum())
print("Recorded nonzero sales:", (sample_series > 0).sum())

In [ ]:
# ---------------------------------------------------------------
# 10. No NaN values in y
# ---------------------------------------------------------------
nan_count = Y_product["y"].isna().sum()
print(nan_count)
print(results)

In [ ]:
results["no_nan_in_y"] = nan_count == 0
print("\nNaN values in y:", nan_count)

In [ ]:
# ---------------------------------------------------------------
# 11. Date range / series length summary
# ---------------------------------------------------------------
print("\nOverall ds range:", Y_product["ds"].min(), "to", Y_product["ds"].max() )
print("\n")
print(Y_product.groupby("unique_id").size().describe())


In [ ]:
results.items()

In [ ]:
S_product 

In [ ]:

# ---------------------------------------------------------------
# Final pass/fail summary
# ---------------------------------------------------------------
print("\n=== STEP 2.1 VALIDATION SUMMARY ===")
for check, passed in results.items():
    print(f"{'PASS' if passed else 'FAIL'} — {check}")

overall = all(results.values())
print(f"\nOVERALL: {'PASS' if overall else 'FAIL'}")
print("Y_product is ready for StatsForecast:", not Y_product[["unique_id", "ds", "y"]].isna().any().any())
print("Reminder: gaps and zero-sale weeks were only counted and reported above — nothing was filled or imputed.")


## 2.2 — Validating the Routing Plan


In [ ]:
# ---------------------------------------------------------------
# 1. Confirm every bottom-level SKU has a routing classification
#    (uses bottom_ids_from_S, already established in Step 2.1)
# ---------------------------------------------------------------
classification_product = pd.read_csv("classification_product.csv")
routed_ids = set(classification_product["unique_id"])
print(bottom_ids_from_S)
print(routed_ids)
missing_routing = bottom_ids_from_S - routed_ids
extra_routing = routed_ids - bottom_ids_from_S
print("SKUs missing a routing classification:", len(missing_routing))
print("Routed SKUs not actually in the bottom level:", len(extra_routing))

In [ ]:
# ---------------------------------------------------------------
# 2. Confirm no SKU has more than one classification row
# ---------------------------------------------------------------
dupe_classification = classification_product["unique_id"].duplicated().sum()
print("SKUs with more than one classification row:", dupe_classification)

In [ ]:
# ---------------------------------------------------------------
# 3. Confirm the classification values are only the 4 expected classes
# ---------------------------------------------------------------
valid_classes = {"smooth", "erratic", "intermittent", "lumpy"}
unexpected_classes = set(classification_product["class"].unique()) - valid_classes
classes_unexpected = valid_classes - set(classification_product["class"].unique()) 
print("Unexpected class labels found:", unexpected_classes)
print("Classes unexpected labels found:", classes_unexpected)


In [ ]:
# ---------------------------------------------------------------
# 4. Print the breakdown — should match your known Step 4.4 result
#    (102 smooth, 39 erratic, 20 intermittent, 61 lumpy)
# ---------------------------------------------------------------
print("\nClassification breakdown:")
print(classification_product["class"].value_counts())


- **`standard_ids`** → these SKUs will be forecasted with **SeasonalNaive / AutoETS** (models that expect fairly continuous, regular data)
- **`intermittent_ids`** → these SKUs will be forecasted with **CrostonClassic / TSB** (models built specifically for sparse, gappy demand, like Talisker, which you found earlier only has real sales in 2 of 57 possible weeks)

In [ ]:
# ---------------------------------------------------------------
# 5. Build the two routing groups that 2.3 will actually use
# ---------------------------------------------------------------
standard_ids = classification_product[classification_product["class"].isin(["smooth", "erratic"])]["unique_id"]
intermittent_ids = classification_product[classification_product["class"].isin(["intermittent", "lumpy"])]["unique_id"]
print(standard_ids)


In [ ]:
print(intermittent_ids)

In [ ]:
print(f"\nStandard group size: {len(standard_ids)}")
print(f"Sparse/intermittent group size: {len(intermittent_ids)}")
print(f"Total: {len(standard_ids) + len(intermittent_ids)} (should equal 222)")

In [ ]:


# ---------------------------------------------------------------
# ADI vs. CV² scatter plot — visualizing the demand classification
# ---------------------------------------------------------------
full_calendar = pd.date_range(Y_product["ds"].min(), Y_product["ds"].max(), freq="W-MON")

def compute_adi_cv2(uid):
    series = Y_product[Y_product["unique_id"] == uid].set_index("ds")["y"].reindex(full_calendar, fill_value=0)
    y = series.values
    nonzero = y[y > 0]
    adi = len(y) / max(1, (y > 0).sum())
    cv2 = (nonzero.std() / nonzero.mean()) ** 2 if len(nonzero) > 1 and nonzero.mean() > 0 else np.nan
    return pd.Series({"adi": adi, "cv2": cv2})

adi_cv2 = classification_product.copy()
adi_cv2[["adi", "cv2"]] = adi_cv2["unique_id"].apply(compute_adi_cv2)

colors = {"smooth": "#1a9850", "erratic": "#fee08b", "intermittent": "#4575b4", "lumpy": "#d73027"}

plt.figure(figsize=(9, 7))
for cls, color in colors.items():
    subset = adi_cv2[adi_cv2["class"] == cls]
    plt.scatter(subset["adi"], subset["cv2"], c=color, label=cls, alpha=0.75, edgecolors="k", linewidths=0.3)

plt.axvline(1.32, color="gray", linestyle="--", linewidth=1)
plt.axhline(0.49, color="gray", linestyle="--", linewidth=1)
plt.xlabel("ADI — Average Demand Interval")
plt.ylabel("CV² — squared coefficient of variation")
plt.title("ADI vs. CV² — Demand Classification (Syntetos & Boylan thresholds)")
plt.legend(title="Class")
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------------
# Final pass/fail
# ---------------------------------------------------------------
step_2_2_pass = (
    len(missing_routing) == 0 and
    len(extra_routing) == 0 and
    dupe_classification == 0 and
    len(unexpected_classes) == 0 and
    (len(standard_ids) + len(intermittent_ids)) == 222
)
print(f"\nSTEP 2.2 OVERALL: {'PASS' if step_2_2_pass else 'FAIL'}")


## 2.3 — Code: Fitting Per Route, With Prediction Intervals


In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, AutoETS, CrostonClassic, CrostonSBA, TSB

# ---------------------------------------------------------------
# 1. Build the two dataframes, using the groups confirmed in 2.2
# ---------------------------------------------------------------
standard_df = Y_product[Y_product["unique_id"].isin(standard_ids)]
intermittent_df = Y_product[Y_product["unique_id"].isin(intermittent_ids)]

print(standard_df)
print("Standard group rows:", standard_df.shape[0], "| series:", standard_df["unique_id"].nunique())
print("Intermittent group rows:", intermittent_df.shape[0], "| series:", intermittent_df["unique_id"].nunique())


In [ ]:
series_lengths = standard_df.groupby("unique_id").size().sort_values()
print(series_lengths.head(141))
print("\nSKUs with fewer than 104 weeks:", (series_lengths < 104).sum())


In [ ]:

# ---------------------------------------------------------------
# 2. Fit the standard group — smooth + erratic SKUs
# ---------------------------------------------------------------
sf_standard = StatsForecast(
    models=[Naive(), SeasonalNaive(season_length=13), AutoETS(season_length=13)],
    freq="W-MON"
)
forecasts_standard = sf_standard.forecast(df=standard_df, h=13, level=[80, 95])
print(forecasts_standard)

In [ ]:
from statsforecast.models import Naive, SeasonalNaive, AutoETS, AutoARIMA, AutoTheta, AutoCES

sf_standard = StatsForecast(
    models=[
        Naive(),
        SeasonalNaive(season_length=13),
        AutoETS(season_length=13),
        AutoARIMA(season_length=13),
        AutoTheta(season_length=13),
        AutoCES(season_length=13),
    ],
    freq="W-MON"
)
forecasts_standard = sf_standard.forecast(df=standard_df, h=13, level=[80, 95])
print(forecasts_standard)

In [ ]:
print(forecasts_standard.columns.tolist())


In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
print(forecasts_standard)


In [ ]:
print(forecasts_standard)


In [ ]:
interval_cols = [c for c in forecasts_standard.columns if "-lo-" in c]
before_counts = (forecasts_standard[interval_cols] < 0).sum()


In [ ]:
# 1. Get a FRESH, unclipped copy to measure the real "before" state
forecasts_standard = sf_standard.forecast(df=standard_df, h=13, level=[80, 95])

interval_cols = [c for c in forecasts_standard.columns if "-lo-" in c]
before_counts = (forecasts_standard[interval_cols] < 0).sum()

# 2. Apply the clip, same as always
forecasts_standard[interval_cols] = forecasts_standard[interval_cols].clip(lower=0)
after_counts = pd.Series(0, index=interval_cols)

# 3. Plot
plt.figure(figsize=(10, 5))
x = range(len(interval_cols))
plt.bar(x, before_counts.values, width=0.4, label="Before clipping", color="#d73027")
plt.bar([i + 0.4 for i in x], after_counts.values, width=0.4, label="After clipping", color="#1a9850")
plt.xticks([i + 0.2 for i in x], interval_cols, rotation=90, fontsize=7)
plt.ylabel("Count of negative values")
plt.title("Negative Confidence-Bound Values — Before vs. After Clipping")
plt.legend()
plt.tight_layout()
plt.show()



## Part 1 — Every model, the columns it produces, and what they mean

| Model | Columns it produces | What the point-forecast column means | What approach the model uses |
|---|---|---|---|
| **Naive** | `Naive`, `Naive-lo-80`, `Naive-hi-80`, `Naive-lo-95`, `Naive-hi-95` | Best guess = whatever the last known actual value was, repeated for every future week | Simplest possible method — no trend, no seasonality, just "assume tomorrow looks like today" |
| **SeasonalNaive** | `SeasonalNaive` + its 4 bound columns | Best guess = whatever happened 13 weeks before this forecasted week (your chosen season length) | Assumes the pattern from last "quarter" repeats |
| **AutoETS** | `AutoETS` + its 4 bound columns | Best guess = a fitted curve combining recent level + trend + seasonal pattern | Automatically searches multiple exponential-smoothing variants, picks the best-fitting one |
| **AutoARIMA** | `AutoARIMA` + its 4 bound columns | Best guess = a fitted autoregressive model, capturing momentum/correlation between recent weeks | Automatically searches different autoregressive structures, picks the best one |
| **AutoTheta** | `AutoTheta` + its 4 bound columns | Best guess = a blend of a straight-line trend and a smoothed curve | Based on the "Theta method" — known for strong performance on shorter series |
| **CES** (from `AutoCES`) | `CES` + its 4 bound columns | Best guess = a more flexible smoothing curve that can adapt its shape over time | A newer smoothing technique, sometimes outperforms classic ETS |

Every model's bound columns mean the same thing, regardless of which model:

- `-lo-80` / `-hi-80` = "80% confident the real value falls between these two numbers"
- `-lo-95` / `-hi-95` = "95% confident" — a wider, more cautious range than the 80% one

In [ ]:
# Just Naive
print(forecasts_standard[["unique_id", "ds", "Naive", "Naive-lo-80", "Naive-hi-80", "Naive-lo-95", "Naive-hi-95"]])



In [ ]:
# Just SeasonalNaive
print(forecasts_standard[["unique_id", "ds", "SeasonalNaive", "SeasonalNaive-lo-80", "SeasonalNaive-hi-80", "SeasonalNaive-lo-95", "SeasonalNaive-hi-95"]])


In [ ]:

# Just AutoETS
print(forecasts_standard[["unique_id", "ds", "AutoETS", "AutoETS-lo-80", "AutoETS-hi-80", "AutoETS-lo-95", "AutoETS-hi-95"]])


In [ ]:

# Just AutoARIMA
print(forecasts_standard[["unique_id", "ds", "AutoARIMA", "AutoARIMA-lo-80", "AutoARIMA-hi-80", "AutoARIMA-lo-95", "AutoARIMA-hi-95"]])


In [ ]:

# Just AutoTheta
print(forecasts_standard[["unique_id", "ds", "AutoTheta", "AutoTheta-lo-80", "AutoTheta-hi-80", "AutoTheta-lo-95", "AutoTheta-hi-95"]])


In [ ]:

# Just CES
print(forecasts_standard[["unique_id", "ds", "CES", "CES-lo-80", "CES-hi-80", "CES-lo-95", "CES-hi-95"]])


In [ ]:
import pandas as pd

holiday_dates = pd.to_datetime([
    "2026-01-01", "2026-04-03", "2026-04-06", "2026-05-01",
    "2026-06-01", "2026-10-20", "2026-12-12", "2026-12-25", "2026-12-26",
])
holiday_weeks = set(d - pd.Timedelta(days=d.weekday()) for d in holiday_dates)
print("Holiday weeks flagged:", sorted(holiday_weeks))

forecasts_standard["is_holiday_week"] = forecasts_standard["ds"].isin(holiday_weeks)
print("\nForecasted weeks that are holiday weeks:")
print(forecasts_standard[forecasts_standard["is_holiday_week"]][["unique_id", "ds"]])

def holiday_uplift_factor(uid, this_holiday_week, full_history):
    last_year_week = this_holiday_week - pd.Timedelta(weeks=52)
    series = full_history[full_history["unique_id"] == uid]
    avg_sales = series["y"].mean()
    last_year_row = series[series["ds"] == last_year_week]
    if last_year_row.empty or avg_sales == 0:
        return 1.0
    return last_year_row["y"].values[0] / avg_sales

uplift_factors = []
for _, row in forecasts_standard.iterrows():
    if row["is_holiday_week"]:
        uplift_factors.append(holiday_uplift_factor(row["unique_id"], row["ds"], Y_product))
    else:
        uplift_factors.append(1.0)

forecasts_standard["holiday_uplift_factor"] = uplift_factors                                  
forecasts_standard["holiday_uplift_factor"] = forecasts_standard["holiday_uplift_factor"].clip(0.5, 2.0)

forecasts_standard["AutoETS_business_adjusted"] = (
    forecasts_standard["AutoETS"] * forecasts_standard["holiday_uplift_factor"]
)

print("\nFinal comparison — original vs. business-adjusted:")
print(forecasts_standard[forecasts_standard["is_holiday_week"]][[
    "unique_id", "ds", "AutoETS", "holiday_uplift_factor", "AutoETS_business_adjusted"
]])


In [ ]:
interval_cols = [c for c in forecasts_standard.columns if "-lo-" in c]
before_counts = (forecasts_standard[interval_cols] < 0).sum()


In [ ]:
interval_cols = [c for c in forecasts_standard.columns if "-lo-" in c]
forecasts_standard[interval_cols] = forecasts_standard[interval_cols].clip(lower=0)


In [ ]:
def compare_step_size(sf, df, h, n_windows, model_cols, step_size_new=13):
    cv_old = sf.cross_validation(df=df, h=h, n_windows=n_windows)                      # default step_size=1
    cv_new = sf.cross_validation(df=df, h=h, n_windows=n_windows, step_size=step_size_new)

    def wape_table(cv):
        errors = cv.copy()
        for m in model_cols:
            errors[f"{m}_abs_err"] = (errors[m] - errors["y"]).abs()
        overall = {m: errors[f"{m}_abs_err"].sum() / errors["y"].abs().sum() for m in model_cols}
        per_product = errors.groupby("unique_id").apply(
            lambda g: pd.Series({m: g[f"{m}_abs_err"].sum() / g["y"].abs().sum() for m in model_cols}),
            include_groups=False
        )
        return pd.Series(overall).sort_values(), per_product.idxmin(axis=1)

    overall_old, winners_old = wape_table(cv_old)
    overall_new, winners_new = wape_table(cv_new)

    print("Products tested — old:", cv_old["unique_id"].nunique(), "| new:", cv_new["unique_id"].nunique())
    print("\nOverall WAPE — step_size=1 (default):")
    print(overall_old)
    print("\nOverall WAPE — step_size=", step_size_new, ":")
    print(overall_new)

    changed = winners_old != winners_new
    print(f"\nProducts whose winning model changed: {changed.sum()} out of {len(changed)}")
    print(pd.DataFrame({"old_winner": winners_old, "new_winner": winners_new})[changed])

    return cv_old, cv_new, winners_old, winners_new


In [ ]:
cv_old, cv_new, winners_old, winners_new = compare_step_size(
    sf_standard, standard_df, h=13, n_windows=2,
    model_cols=["Naive", "SeasonalNaive", "AutoETS", "AutoARIMA", "AutoTheta", "CES"]
)


In [ ]:
cv_results = sf_standard.cross_validation(df=standard_df, h=13, n_windows=2, step_size=13)
cv_results

In [ ]:
model_cols = ["Naive", "SeasonalNaive", "AutoETS", "AutoARIMA", "AutoTheta", "CES"]

errors = cv_results.copy()
for m in model_cols:
    errors[f"{m}_abs_err"] = (errors[m] - errors["y"]).abs()

# Overall accuracy per model, across everything
mae_overall = {m: errors[f"{m}_abs_err"].mean() for m in model_cols}
print(pd.Series(mae_overall).sort_values())

# Accuracy per model, per specific product
mae_per_product = errors.groupby("unique_id")[[f"{m}_abs_err" for m in model_cols]].mean()
mae_per_product.columns = model_cols
best_model_per_product = mae_per_product.idxmin(axis=1)
print(best_model_per_product.value_counts())


In [ ]:
wape_overall = {m: errors[f"{m}_abs_err"].sum() / errors["y"].abs().sum() for m in model_cols}
print(pd.Series(wape_overall).sort_values())

wape_per_product = errors.groupby("unique_id").apply(
    lambda g: pd.Series({m: g[f"{m}_abs_err"].sum() / g["y"].abs().sum() for m in model_cols})
)
best_model_per_product_wape = wape_per_product.idxmin(axis=1)
print(best_model_per_product)
print(best_model_per_product_wape.value_counts())


In [ ]:
wape_series = pd.Series(wape_overall).sort_values()

plt.figure(figsize=(8, 5))
colors = ["#1a9850" if v == wape_series.min() else "#4575b4" for v in wape_series]
plt.bar(wape_series.index, wape_series.values, color=colors, edgecolor="black")
plt.ylabel("WAPE")
plt.title("Standard Group — Overall Accuracy per Model (lower is better)")
for i, v in enumerate(wape_series.values):
    plt.text(i, v + 0.003, f"{v:.1%}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
print(best_model_per_product[best_model_per_product == "AutoARIMA"])


In [ ]:
print(best_model_per_product[best_model_per_product == "AutoTheta"])


In [ ]:
print(best_model_per_product[best_model_per_product == "AutoETS"])


In [ ]:
print(best_model_per_product[best_model_per_product == "CES"])


In [ ]:
print(best_model_per_product[best_model_per_product == "Naive"])


In [ ]:
print(best_model_per_product[best_model_per_product == "SeasonalNaive"])


In [ ]:
final_forecast = forecasts_standard.copy()
final_forecast["chosen_model_forecast"] = final_forecast.apply(
    lambda row: row[best_model_per_product[row["unique_id"]]], axis=1
)


In [ ]:
print(final_forecast[["unique_id", "ds", "chosen_model_forecast"]])


In [ ]:
final_forecast

In [ ]:
final_forecast.to_csv("../eabl parsed csv/final_forecast_standard.csv", index=False)


In [ ]:
chosen_abs_err = errors.apply(
    lambda row: row[f"{best_model_per_product[row['unique_id']]}_abs_err"], axis=1
)
blended_wape = chosen_abs_err.sum() / errors["y"].abs().sum()
print("Blended per-SKU WAPE:", blended_wape)


In [ ]:
# Clip negatives
final_forecast["chosen_model_forecast"] = final_forecast["chosen_model_forecast"].clip(lower=0)

# Migrate holiday adjustment, with the sanity clip applied
forecasts_standard["holiday_uplift_factor"] = forecasts_standard["holiday_uplift_factor"].clip(0.5, 2.0)
final_forecast["chosen_model_forecast_adjusted"] = (
    final_forecast["chosen_model_forecast"] * forecasts_standard["holiday_uplift_factor"]
)


In [ ]:
print(final_forecast[["unique_id", "ds", "chosen_model_forecast", "chosen_model_forecast_adjusted"]].head(10))



# ---------------------------------------------------------------
# 2. Fit the standard group — smooth + erratic SKUs while converting y to log then converting it back
# ---------------------------------------------------------------

In [ ]:
import numpy as np

# ---------------------------------------------------------------
# 1. Convert y to log form, before fitting anything
# ---------------------------------------------------------------
standard_df_log = standard_df.copy()
standard_df_log["y"] = np.log1p(standard_df_log["y"])  # log(1 + y) — safely handles zero-sale weeks

# ---------------------------------------------------------------
# 2. Fit the SAME 6 models, just on the log-converted data
# ---------------------------------------------------------------
from statsforecast.models import Naive, SeasonalNaive, AutoETS, AutoARIMA, AutoTheta, AutoCES

sf_standard_log = StatsForecast(
    models=[
        Naive(),
        SeasonalNaive(season_length=13),
        AutoETS(season_length=13),
        AutoARIMA(season_length=13),
        AutoTheta(season_length=13),
        AutoCES(season_length=13),
    ],
    freq="W-MON"
)



In [ ]:
# ---------------------------------------------------------------
# 3. Real future forecast (log scale), then convert back
# ---------------------------------------------------------------
forecasts_log = sf_standard_log.forecast(df=standard_df_log, h=13, level=[80, 95])

model_cols = ["Naive", "SeasonalNaive", "AutoETS", "AutoARIMA", "AutoTheta", "CES"]
for m in model_cols:
    forecasts_log[m] = np.expm1(forecasts_log[m])  # convert back to real sales numbers


In [ ]:
cv_old_log = sf_standard_log.cross_validation(df=standard_df_log, h=13, n_windows=2)
cv_new_log = sf_standard_log.cross_validation(df=standard_df_log, h=13, n_windows=2, step_size=13)

for cv in (cv_old_log, cv_new_log):
    for m in model_cols:
        cv[m] = np.expm1(cv[m])
    cv["y"] = np.expm1(cv["y"])

def wape_table_log(cv):
    errors = cv.copy()
    for m in model_cols:
        errors[f"{m}_abs_err"] = (errors[m] - errors["y"]).abs()
    overall = {m: errors[f"{m}_abs_err"].sum() / errors["y"].abs().sum() for m in model_cols}
    return pd.Series(overall).sort_values()

print("Log-scale WAPE — step_size=1:")
print(wape_table_log(cv_old_log))
print("\nLog-scale WAPE — step_size=13:")
print(wape_table_log(cv_new_log))


In [ ]:

# ---------------------------------------------------------------
# 4. Cross-validate on the log scale, then convert back, same as step 3
# ---------------------------------------------------------------
cv_results_log = sf_standard_log.cross_validation(df=standard_df_log, h=13, n_windows=2, step_size=13)

for m in model_cols:
    cv_results_log[m] = np.expm1(cv_results_log[m])
cv_results_log["y"] = np.expm1(cv_results_log["y"])  # convert the real actuals back too



In [ ]:
# ---------------------------------------------------------------
# 5. Compute WAPE, same formula as before, for a fair comparison
# ---------------------------------------------------------------
errors_log = cv_results_log.copy()
for m in model_cols:
    errors_log[f"{m}_abs_err"] = (errors_log[m] - errors_log["y"]).abs()

wape_overall_log = {m: errors_log[f"{m}_abs_err"].sum() / errors_log["y"].abs().sum() for m in model_cols}
print("Log-scale WAPE per model:")
print(pd.Series(wape_overall_log).sort_values())


In [ ]:
# ---------------------------------------------------------------
# 6. Pick the best model per product, same as before, on the log version
#    (include_groups moved onto .apply(), where it actually belongs —
#    this is the fix for the TypeError you hit)
# ---------------------------------------------------------------
try:
    wape_per_product_log = errors_log.groupby("unique_id").apply(
        lambda g: pd.Series({m: g[f"{m}_abs_err"].sum() / g["y"].abs().sum() for m in model_cols}),
        include_groups=False
    )
except TypeError:
    # Fallback for older pandas versions that don't support include_groups at all
    wape_per_product_log = errors_log.groupby("unique_id").apply(
        lambda g: pd.Series({m: g[f"{m}_abs_err"].sum() / g["y"].abs().sum() for m in model_cols})
    )

best_model_per_product_log = wape_per_product_log.idxmin(axis=1)

chosen_abs_err_log = errors_log.apply(
    lambda row: row[f"{best_model_per_product_log[row['unique_id']]}_abs_err"], axis=1
)
blended_wape_log = chosen_abs_err_log.sum() / errors_log["y"].abs().sum()

print(f"\nLog-scale blended per-SKU WAPE: {blended_wape_log:.4f}")
print(f"Raw-scale blended per-SKU WAPE (already have): 0.1537")

# Sparse / Intermittent Group — Start of Section

This section forecasts the **81 "intermittent + lumpy" SKUs** (the other half of the Product hierarchy's 222 SKUs, routed here instead of the standard group because they have sparse, gappy sales histories).

## The flow this section follows

1. **Measure how much real data each of the 81 products has** (row counts) — reveals most are very short on history
2. **Split into 3 confidence tiers** based on that row count:
   - Tier 1 (27 SKUs): enough data for the full 2-window cross-validation test
   - Tier 2 (16 SKUs): enough for only 1 test window
   - Tier 3 (38 SKUs): too little data to test at all — gets a safe default forecast instead
3. **Fit 6 sparse-demand models** (CrostonClassic, CrostonOptimized, CrostonSBA, TSB, ADIDA, IMAPA) — none require a season_length or minimum-history precondition
4. **Test each tier honestly** with cross-validation (where possible), measure WAPE per model per product, pick each product's own best-tested model
5. **Produce the real future forecast** and combine all 3 tiers into one final `chosen_model_forecast` column, each row flagged `tested=True/False` for honesty
6. **Combine with the standard-group forecast** into the full 222-SKU table, ready for F3 (reconciliation)

## Where we are right now

Steps 1–4 are done for Tier 1 (winner-selection complete) and partially done for Tier 2/3. **Still outstanding:** finalize Tier 1's chosen forecast + confidence flag, run Tier 2's WAPE/winner-selection, finalize Tier 3's default forecast, then combine all three tiers and merge with the standard group.


In [ ]:
series_lengths_sparse = intermittent_df.groupby("unique_id").size().sort_values()
print(series_lengths_sparse.head(20))
print("\nSeries with fewer than 27 rows:", (series_lengths_sparse < 27).sum())



In [ ]:
import matplotlib.pyplot as plt

# ---------------------------------------------------------------
# Histogram of row-counts per sparse product — showing the 3 tiers visually
# ---------------------------------------------------------------
plt.figure(figsize=(10, 5))
plt.hist(series_lengths_sparse, bins=30, color="#4575b4", edgecolor="black", alpha=0.8)

plt.axvline(14, color="#d73027", linestyle="--", linewidth=1.5, label="14 rows — min for 1 test window")
plt.axvline(26, color="#1a9850", linestyle="--", linewidth=1.5, label="27 rows — min for 2 test windows")

plt.axvspan(0, 14, color="#d73027", alpha=0.08)
plt.axvspan(14, 26, color="#fee08b", alpha=0.08)
plt.axvspan(26, series_lengths_sparse.max() + 5, color="#1a9850", alpha=0.08)

plt.text(7, plt.ylim()[1]*0.9, "Tier 3\n(38 products)\nuntestable", ha="center", fontsize=9)
plt.text(20, plt.ylim()[1]*0.9, "Tier 2\n(16 products)\n1 window", ha="center", fontsize=9)
plt.text(40, plt.ylim()[1]*0.9, "Tier 1\n(27 products)\n2 windows", ha="center", fontsize=9)

plt.xlabel("Number of real sales rows per product")
plt.ylabel("Number of products")
plt.title("Row-Count Distribution Across the 81 Sparse Products — Tier Boundaries")
plt.legend()
plt.tight_layout()
plt.show()


## Splitting the Sparse Group into 3 Confidence Tiers

Before fitting or testing anything, each of the 81 sparse products is sorted purely by
**how many real rows of sales history it has** — not by category, brand, or anything
else. This determines whether a product has enough data to support a full, honest
cross-validation test, a partial one, or no test at all:

- **Fewer than 14 rows** → Tier 3: too little data to run even one 13-week test window
- **14–25 rows** → Tier 2: enough for one honest test window
- **26+ rows** → Tier 1: enough for the full, 2-window rigorous test

This split is re-run at several points in this notebook (whenever the underlying variables
need rebuilding) — this is the original cell where the tier counts were first established
and confirmed: **38 Tier 3, 16 Tier 2, 27 Tier 1.**


In [ ]:
# Only needed if intermittent_df itself is also missing — check first:
# print(intermittent_df.shape)

series_lengths_sparse = intermittent_df.groupby("unique_id").size().sort_values()

too_short_for_any_cv = series_lengths_sparse[series_lengths_sparse < 14]
testable_1_window = series_lengths_sparse[(series_lengths_sparse >= 14) & (series_lengths_sparse < 27)]
testable_2_windows = series_lengths_sparse[series_lengths_sparse >= 27]

ids_2w = testable_2_windows.index
intermittent_df_2w = intermittent_df[intermittent_df["unique_id"].isin(ids_2w)]
print("NEW Tier 1:", len(testable_2_windows), "(was 27 before)")
print("NEW Tier 2:", len(testable_1_window), "(was 16 before)")
print("NEW Tier 3:", len(too_short_for_any_cv), "(was 38 before)")

In [ ]:
near_boundary = series_lengths_sparse[
    (series_lengths_sparse >= 10) & (series_lengths_sparse <= 32)
]
print("Products within range of a tier boundary (worth checking by eye):")
print(near_boundary)


In [ ]:
testable_2_windows.shape

In [ ]:
print("Current Tier 1 count:", len(testable_2_windows))
print("Current Tier 2 count:", len(testable_1_window))
print("Current Tier 3 count:", len(too_short_for_any_cv))
print("Expected if these are OLD (pre-fix) values: 27, 16, 38")
print("If these numbers DON'T match 27/16/38, these are already the NEW values — the old ones are gone")


In [ ]:
try:
    print(intermittent_df.shape)
except NameError:
    print("intermittent_df is gone — need to rebuild from Y_product")

try:
    print(testable_2_windows)
except NameError:
    print("testable_2_windows is gone — need to re-run the series-length check")


### Tier 1 — 27 SKUs with enough data for the full, rigorous test (n_windows=2)

These are the sparse products with 26+ rows of real history — enough to support the full
2-window cross-validation test, same rigor as the standard group. Tested against all 6
sparse-demand models (`CrostonClassic`, `CrostonOptimized`, `CrostonSBA`, `TSB`, `ADIDA`, `IMAPA`),
with each product assigned whichever model proved most accurate for it specifically.

**Note:** `TSB`'s smoothing parameters were later re-tuned (`alpha_d=0.2 → 0.3`) and this
tier's results were re-tested and re-confirmed afterward — the winners below reflect that
updated, final version.


In [ ]:
import pandas as pd
import numpy as np
from statsforecast import StatsForecast
from statsforecast.models import TSB

# Build a synthetic series of each length and see where it stops failing
for length in range(24, 30):
    dates = pd.date_range("2024-01-01", periods=length, freq="W-MON")
    test_df = pd.DataFrame({
        "unique_id": ["test"] * length,
        "ds": dates,
        "y": np.random.rand(length),
    })
    sf_test = StatsForecast(models=[TSB(alpha_d=0.3, alpha_p=0.3)], freq="W-MON")
    try:
        sf_test.cross_validation(df=test_df, h=13, n_windows=2, step_size=13)
        print(f"length={length}: SUCCESS")
    except ValueError as e:
        print(f"length={length}: FAILS")


In [ ]:
tsb_variants = [
    TSB(alpha_d=0.1, alpha_p=0.1, alias="TSB_01"),
    TSB(alpha_d=0.2, alpha_p=0.2, alias="TSB_02"),
    TSB(alpha_d=0.3, alpha_p=0.3, alias="TSB_03"),
    TSB(alpha_d=0.5, alpha_p=0.5, alias="TSB_05"),
]

sf_tsb_test = StatsForecast(models=tsb_variants, freq="W-MON")
cv_tsb_test = sf_tsb_test.cross_validation(df=intermittent_df_2w, h=13, n_windows=2,step_size=13)
print(sf_tsb_test)
print(cv_tsb_test)

In [ ]:
tsb_cols = ["TSB_01", "TSB_02", "TSB_03", "TSB_05"]
errors_tsb = cv_tsb_test.copy()
for m in tsb_cols:
    errors_tsb[f"{m}_abs_err"] = (errors_tsb[m] - errors_tsb["y"]).abs()

wape_tsb = {m: errors_tsb[f"{m}_abs_err"].sum() / errors_tsb["y"].abs().sum() for m in tsb_cols}
print(pd.Series(wape_tsb).sort_values())


In [ ]:
tsb_cols = ["TSB_01", "TSB_03", "TSB_05"]
errors_tsb = cv_tsb_test.copy()
for m in tsb_cols:
    errors_tsb[f"{m}_abs_err"] = (errors_tsb[m] - errors_tsb["y"]).abs()

wape_tsb = {m: errors_tsb[f"{m}_abs_err"].sum() / errors_tsb["y"].abs().sum() for m in tsb_cols}
print(pd.Series(wape_tsb).sort_values())


In [ ]:
alpha_values = [0.1, 0.2, 0.3, 0.5]
wape_values = [0.794014, 0.767265, 0.763222, 0.787423]

plt.figure(figsize=(8, 5))
plt.plot(alpha_values, wape_values, marker="o", color="#4575b4", linewidth=2)
best_idx = wape_values.index(min(wape_values))
plt.scatter(alpha_values[best_idx], wape_values[best_idx], color="#1a9850", s=120, zorder=5, label="Best (alpha=0.3)")
plt.xlabel("alpha_d / alpha_p")
plt.ylabel("WAPE")
plt.title("TSB Sensitivity — WAPE vs. Alpha Setting")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 1. Update sf_intermittent with the new alpha value
from statsforecast import StatsForecast
from statsforecast.models import CrostonClassic, CrostonOptimized, CrostonSBA, TSB, ADIDA, IMAPA

sf_intermittent = StatsForecast(
    models=[
        CrostonClassic(),
        CrostonOptimized(),
        CrostonSBA(),
        TSB(alpha_d=0.3, alpha_p=0.3),   # ← updated value applied here
        ADIDA(),
        IMAPA(),
    ],
    freq="W-MON"
)


In [ ]:
intermittent_df.shape

In [ ]:
# 2. Re-run cross-validation for Tier 1 with the updated TSB setting
ids_2w = testable_2_windows.index
intermittent_df_2w = intermittent_df[intermittent_df["unique_id"].isin(ids_2w)]
cv_2w = sf_intermittent.cross_validation(df=intermittent_df_2w, h=13, n_windows=2, step_size=13)

print(cv_2w)


In [ ]:
intermittent_df_2w.shape

In [ ]:
# 3. Recompute WAPE per product, using the updated cv_2w

sparse_model_cols = ["CrostonClassic", "CrostonOptimized", "CrostonSBA", "TSB", "ADIDA", "IMAPA"]

errors_2w = cv_2w.copy()
for m in sparse_model_cols:
    errors_2w[f"{m}_abs_err"] = (errors_2w[m] - errors_2w["y"]).abs()

wape_per_product_2w = errors_2w.groupby("unique_id").apply(
    lambda g: pd.Series({m: g[f"{m}_abs_err"].sum() / g["y"].abs().sum() for m in sparse_model_cols}),
    include_groups=False
)
print(wape_per_product_2w.loc["Beer/Mainstream/Guinness Smooth/796716"])


In [ ]:
errors_2w = cv_2w.copy()
for m in sparse_model_cols:
    errors_2w[f"{m}_abs_err"] = (errors_2w[m] - errors_2w["y"]).abs()

wape_per_product_2w = errors_2w.groupby("unique_id").apply(
    lambda g: pd.Series({m: g[f"{m}_abs_err"].sum() / g["y"].abs().sum() for m in sparse_model_cols}),
    include_groups=False
)
best_model_per_product_2w = wape_per_product_2w.idxmin(axis=1)

print("WAPE per product (Tier 1):")
print(wape_per_product_2w)


In [ ]:
cv_old_2w, cv_new_2w, winners_old_2w, winners_new_2w = compare_step_size(
    sf_intermittent, intermittent_df_2w, h=13, n_windows=2,
    model_cols=sparse_model_cols
)


In [ ]:
print(wape_per_product_2w)


In [ ]:
# 4. Re-pick each product's winner — this may now differ from before
best_model_per_product_2w = wape_per_product_2w.idxmin(axis=1)
print(best_model_per_product_2w)

In [ ]:

counts = best_model_per_product_2w.value_counts()

plt.figure(figsize=(8, 5))
plt.bar(counts.index, counts.values, color="#4575b4", edgecolor="black")
plt.ylabel("Number of products won")
plt.title("Tier 1 — Which Model Wins How Many Products (out of 25)")
for i, v in enumerate(counts.values):
    plt.text(i, v + 0.3, str(v), ha="center")
plt.tight_layout()
plt.show()


In [ ]:
# 5. Rebuild the real forecast and final chosen_model_forecast using the updated winners
forecasts_intermittent = sf_intermittent.forecast(df=intermittent_df, h=13)
forecast_2w = forecasts_intermittent[forecasts_intermittent["unique_id"].isin(ids_2w)]
final_2w = forecast_2w.copy()
print(forecasts_intermittent);
print(forecast_2w);

final_2w["chosen_model_forecast"] = final_2w.apply(
    lambda row: row[best_model_per_product_2w[row["unique_id"]]], axis=1
)
final_2w["chosen_model_forecast"] = final_2w["chosen_model_forecast"].clip(lower=0)
print("\n")
print(final_2w)
print(final_2w["chosen_model_forecast"])

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(20, 11))

# ---------------------------------------------------------------
# Panel 1 — Standard group: WAPE per model
# ---------------------------------------------------------------
wape_series = pd.Series(wape_overall).sort_values()
colors = ["#1a9850" if v == wape_series.min() else "#4575b4" for v in wape_series]
axes[0, 0].bar(wape_series.index, wape_series.values, color=colors, edgecolor="black")
axes[0, 0].set_title("Standard Group — WAPE per Model")
axes[0, 0].set_ylabel("WAPE")
axes[0, 0].tick_params(axis="x", rotation=45)
for i, v in enumerate(wape_series.values):
    axes[0, 0].text(i, v + 0.003, f"{v:.1%}", ha="center", fontsize=8)

# ---------------------------------------------------------------
# Panel 2 — Standard group: winning-model distribution (141 products)
# ---------------------------------------------------------------
std_counts = best_model_per_product.value_counts()
axes[0, 1].bar(std_counts.index, std_counts.values, color="#4575b4", edgecolor="black")
axes[0, 1].set_title("Standard Group — Winning Model Distribution (141 SKUs)")
axes[0, 1].set_ylabel("Products won")
axes[0, 1].tick_params(axis="x", rotation=45)
for i, v in enumerate(std_counts.values):
    axes[0, 1].text(i, v + 0.5, str(v), ha="center", fontsize=8)

# ---------------------------------------------------------------
# Panel 3 — Overall blended WAPE: standard group result vs. project target
# ---------------------------------------------------------------
blended_values = {"Blended (per-SKU)": 0.1537, "Project target": 0.1564, "Best single model": wape_series.min()}
bar_colors = ["#1a9850", "#999999", "#d73027"]
axes[0, 2].bar(blended_values.keys(), blended_values.values(), color=bar_colors, edgecolor="black")
axes[0, 2].set_title("Standard Group — Blended Result vs. Target")
axes[0, 2].set_ylabel("WAPE")
axes[0, 2].tick_params(axis="x", rotation=20)
for i, (k, v) in enumerate(blended_values.items()):
    axes[0, 2].text(i, v + 0.003, f"{v:.1%}", ha="center", fontsize=8)

# ---------------------------------------------------------------
# Panel 4 — Sparse Tier 1: WAPE per model (recomputed fresh from cv_2w)
# ---------------------------------------------------------------
errors_2w_check = cv_2w.copy()
for m in sparse_model_cols:
    errors_2w_check[f"{m}_abs_err"] = (errors_2w_check[m] - errors_2w_check["y"]).abs()
wape_sparse_series = pd.Series({
    m: errors_2w_check[f"{m}_abs_err"].sum() / errors_2w_check["y"].abs().sum() for m in sparse_model_cols
}).sort_values()
colors_sparse = ["#1a9850" if v == wape_sparse_series.min() else "#4575b4" for v in wape_sparse_series]
axes[1, 0].bar(wape_sparse_series.index, wape_sparse_series.values, color=colors_sparse, edgecolor="black")
axes[1, 0].set_title("Sparse Tier 1 — WAPE per Model (27 SKUs)")
axes[1, 0].set_ylabel("WAPE")
axes[1, 0].tick_params(axis="x", rotation=45)

# ---------------------------------------------------------------
# Panel 5 — Sparse Tier 1: winning-model distribution
# ---------------------------------------------------------------
sparse_counts = best_model_per_product_2w.value_counts()
axes[1, 1].bar(sparse_counts.index, sparse_counts.values, color="#4575b4", edgecolor="black")
axes[1, 1].set_title("Sparse Tier 1 — Winning Model Distribution (27 SKUs)")
axes[1, 1].set_ylabel("Products won")
axes[1, 1].tick_params(axis="x", rotation=45)
for i, v in enumerate(sparse_counts.values):
    axes[1, 1].text(i, v + 0.3, str(v), ha="center", fontsize=8)

# ---------------------------------------------------------------
# Panel 6 — Sparse group: row-count distribution and tier boundaries
# ---------------------------------------------------------------
axes[1, 2].hist(series_lengths_sparse, bins=25, color="#4575b4", edgecolor="black", alpha=0.8)
axes[1, 2].axvline(14, color="#d73027", linestyle="--", label="14 rows (Tier 2 min)")
axes[1, 2].axvline(26, color="#1a9850", linestyle="--", label="26 rows (Tier 1 min)")
axes[1, 2].set_title("Sparse Group — Row-Count Distribution (81 SKUs)")
axes[1, 2].set_xlabel("Rows per product")
axes[1, 2].legend(fontsize=8)

plt.suptitle("F1 Forecasting — Full Summary Dashboard", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
flagged_products = [
    "Spirits/Reserve/Singleton/734970",
    "Spirits/Premium/Gordons Flavours/768300",
    "Spirits/Reserve/Glenkinchie/624151",
]

for product in flagged_products:
    sample = Y_product[Y_product["unique_id"] == product].sort_values("y", ascending=False)
    print(f"\n--- {product} ---")
    print(sample.head(5))
    print("Average weekly sales:", sample["y"].mean())


In [ ]:
chosen_abs_err_2w = errors_2w.apply(
    lambda row: row[f"{best_model_per_product_2w[row['unique_id']]}_abs_err"], axis=1
)
blended_wape_2w = chosen_abs_err_2w.sum() / errors_2w["y"].abs().sum()
print("Tier 1 blended per-SKU WAPE:", blended_wape_2w)


In [ ]:
def compare_weekly_vs_cumulative(cv_df, model_cols):
    results = []
    for uid, group in cv_df.groupby("unique_id"):
        actual_total = group["y"].sum()
        for m in model_cols:
            weekly_wape = (group[m] - group["y"]).abs().sum() / group["y"].abs().sum()
            cumulative_error = abs(group[m].sum() - actual_total) / actual_total if actual_total > 0 else None
            results.append({
                "unique_id": uid,
                "model": m,
                "actual_total": round(actual_total, 2),
                "weekly_wape": weekly_wape,
                "weekly_wape_pct": round(weekly_wape * 100, 2) if weekly_wape is not None else None,
                "cumulative_error": cumulative_error,
                "cumulative_error_pct": round(cumulative_error * 100, 2) if cumulative_error is not None else None,
            })
    return pd.DataFrame(results)

comparison_2w = compare_weekly_vs_cumulative(cv_2w, sparse_model_cols)
comparison_2w.to_csv("../eabl parsed csv/tier1_weekly_vs_cumulative.csv", index=False)
print(comparison_2w)


In [ ]:
comparison_2w = compare_weekly_vs_cumulative(cv_2w, sparse_model_cols)
comparison_2w.to_csv("../eabl parsed csv/tier1_weekly_vs_cumulative.csv", index=False)
print(comparison_2w)

In [ ]:
print("Rows for this product now:", series_lengths_sparse.get("Spirits/Premium/JW Red Label/763938", "NOT IN intermittent_df AT ALL"))
print("Is it in too_short_for_any_cv (Tier 3)?", "Spirits/Premium/JW Red Label/763938" in too_short_for_any_cv.index)


In [ ]:
# Tier 1 overall WAPE (using each product's own winning model)
tier1_errors = []
for uid, model in best_model_per_product_2w.items():
    product_rows = cv_2w[cv_2w["unique_id"] == uid]
    abs_err = (product_rows[model] - product_rows["y"]).abs().sum()
    actual = product_rows["y"].abs().sum()
    tier1_errors.append({"abs_err": abs_err, "actual": actual})

tier1_df = pd.DataFrame(tier1_errors)
tier1_overall_wape = tier1_df["abs_err"].sum() / tier1_df["actual"].sum()
print(f"Tier 1 overall WAPE: {tier1_overall_wape:.2%}")

### Tier 2 — 16 SKUs that can only support one test window

Same evaluation process as Tier 1, but with `n_windows=1` instead of `2`, since these products
only have 14–25 rows of real history — enough for one honest test, not two.


In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import CrostonClassic, CrostonOptimized, CrostonSBA, TSB, ADIDA, IMAPA

sf_intermittent = StatsForecast(
    models=[
        CrostonClassic(),
        CrostonOptimized(),
        CrostonSBA(),
        TSB(alpha_d=0.3, alpha_p=0.3),   # already updated from the Tier 1 tuning test
        ADIDA(),
        IMAPA(),
    ],
    freq="W-MON"
)


In [ ]:
# ---------------------------------------------------------------
# Tier 2 — test, score, and pick winners for the 18 one-window SKUs
# ---------------------------------------------------------------
ids_1w = testable_1_window.index
intermittent_df_1w = intermittent_df[intermittent_df["unique_id"].isin(ids_1w)]
cv_1w = sf_intermittent.cross_validation(df=intermittent_df_1w, h=13, n_windows=1)

assert len(testable_1_window) == 18, f"Expected 18, got {len(testable_1_window)}"
print("Tier 2 confirmed:", len(ids_1w), "products")

print(ids_1w)
print(intermittent_df_1w)
print(cv_1w)


In [ ]:
errors_1w = cv_1w.copy()
for m in sparse_model_cols:
    errors_1w[f"{m}_abs_err"] = (errors_1w[m] - errors_1w["y"]).abs()

wape_per_product_1w = errors_1w.groupby("unique_id").apply(
    lambda g: pd.Series({m: g[f"{m}_abs_err"].sum() / g["y"].abs().sum() for m in sparse_model_cols}),
    include_groups=False
)
best_model_per_product_1w = wape_per_product_1w.idxmin(axis=1)

print("WAPE per product (Tier 2):")
print(wape_per_product_1w)
print("\nWinning model per product (Tier 2):")
print(best_model_per_product_1w)


In [ ]:
tier1_flagged = [
    "Spirits/Mainstream/Kenya Cane/717336",
    "Spirits/Mainstream/Kenya Cane/717383",
    "Spirits/Premium/Gordons Flavours/768493",
    "Spirits/Premium/JW Red Label/763938",
    "Spirits/Reserve/Singleton/734970",
]

for uid in tier1_flagged:
    if uid in best_model_per_product_2w:
        winning_model = best_model_per_product_2w[uid]
        row = comparison_2w[(comparison_2w["unique_id"] == uid) & (comparison_2w["model"] == winning_model)]
        print(f"[Tier 1] {uid}")
        print(row[["unique_id", "model", "weekly_wape_pct", "cumulative_error_pct"]])
    elif uid in best_model_per_product_1w:
        winning_model = best_model_per_product_1w[uid]
        row = comparison_1w[(comparison_1w["unique_id"] == uid) & (comparison_1w["model"] == winning_model)]
        print(f"[Tier 2 — reclassified from Tier 1] {uid}")
        print(row[["unique_id", "model", "weekly_wape_pct", "cumulative_error_pct"]])
    else:
        print(f"[NOT FOUND] {uid} — check Tier 3 or verify it still exists in intermittent_df")
print("cv_2w shape:", cv_2w.shape)
print("sparse_model_cols:", sparse_model_cols)

In [ ]:
# ---------------------------------------------------------------
# Build the real forecast and final chosen_model_forecast for Tier 2
# ---------------------------------------------------------------
forecast_1w = forecasts_intermittent[forecasts_intermittent["unique_id"].isin(ids_1w)]
final_1w = forecast_1w.copy()
final_1w["chosen_model_forecast"] = final_1w.apply(
    lambda row: row[best_model_per_product_1w[row["unique_id"]]], axis=1
)
final_1w["chosen_model_forecast"] = final_1w["chosen_model_forecast"].clip(lower=0)

print("\nFinal Tier 2 forecast sample:")
print(final_1w[["unique_id", "ds", "chosen_model_forecast"]].head(10))


In [ ]:
# ---------------------------------------------------------------
# Flag known high-risk products in BOTH tiers
# ---------------------------------------------------------------
low_confidence_ids = {
    "Spirits/Reserve/Singleton/734970",              # Tier 1
    "Spirits/Premium/Gordons Flavours/768300",        # Tier 2
    "Spirits/Reserve/Glenkinchie/624151",             # Tier 2
}

final_2w["low_confidence"] = final_2w["unique_id"].isin(low_confidence_ids)
final_1w["low_confidence"] = final_1w["unique_id"].isin(low_confidence_ids)

print("Tier 1 flagged rows:", final_2w["low_confidence"].sum())
print("Tier 2 flagged rows:", final_1w["low_confidence"].sum())

print("\nTier 1 flagged products:", final_2w.loc[final_2w["low_confidence"], "unique_id"].unique())
print("Tier 2 flagged products:", final_1w.loc[final_1w["low_confidence"], "unique_id"].unique())


In [ ]:
flagged_products = [
    "Spirits/Reserve/Singleton/734970",
    "Spirits/Premium/Gordons Flavours/768300",
    "Spirits/Reserve/Glenkinchie/624151",
]

for product in flagged_products:
    sample = Y_product[Y_product["unique_id"] == product].sort_values("y", ascending=False)
    print(f"\n--- {product} ---")
    print(sample.head(5))
    print("Average weekly sales:", sample["y"].mean())


In [ ]:
flagged_products = [
    "Spirits/Reserve/Singleton/734970",
    "Spirits/Premium/Gordons Flavours/768300",
    "Spirits/Reserve/Glenkinchie/624151",
]

buffer_recommendations = []
for product in flagged_products:
    sales = Y_product[Y_product["unique_id"] == product]["y"]
    min_level = sales.mean()                  # the floor — average demand
    max_level = sales.quantile(0.90)           # the ceiling — 90th percentile, not the raw max
    buffer_recommendations.append({
        "unique_id": product,
        "recommended_min": round(min_level, 2),
        "recommended_max": round(max_level, 2),
    })

buffer_df = pd.DataFrame(buffer_recommendations)
print(buffer_df)


In [ ]:
buffer_df.to_csv("../eabl parsed csv/low_confidence_buffer_recommendations.csv", index=False)
buffer_df

In [ ]:
final_2w = final_2w.merge(buffer_df, on="unique_id", how="left")
final_1w = final_1w.merge(buffer_df, on="unique_id", how="left")


In [ ]:
def build_combined_forecast(wape_per_product, forecast_df, model_cols, top_n=2):
    results = []
    for uid in wape_per_product.index:
        top_models = wape_per_product.loc[uid].nsmallest(top_n)       # the 2 best-scoring models for this product
        weights = (1 / top_models) / (1 / top_models).sum()           # better (lower) error = more weight
        row_forecasts = forecast_df[forecast_df["unique_id"] == uid].copy()
        row_forecasts["chosen_model_forecast"] = sum(
            row_forecasts[m] * weights[m] for m in top_models.index    # weighted blend of the top 2
        )
        results.append(row_forecasts)
    return pd.concat(results, ignore_index=True)

final_1w_combined = build_combined_forecast(wape_per_product_1w, forecast_1w, sparse_model_cols, top_n=2)
final_1w_combined["chosen_model_forecast"] = final_1w_combined["chosen_model_forecast"].clip(lower=0)
pd.set_option('display.max_rows', None)
print(final_1w_combined[["unique_id", "ds", "chosen_model_forecast"]])

print(final_1w_combined[["unique_id", "ds", "chosen_model_forecast"]])


In [ ]:
pd.set_option('display.max_rows', None)
print(final_1w_combined[["unique_id", "ds", "chosen_model_forecast"]])
pd.reset_option('display.max_rows')


In [ ]:
def build_combined_forecast_for_testing(wape_per_product, cv_df, model_cols, top_n=2):
    results = []
    for uid, group in cv_df.groupby("unique_id"):
        top_models = wape_per_product.loc[uid].nsmallest(top_n)
        weights = (1 / top_models) / (1 / top_models).sum()
        group = group.copy()
        group["blended_forecast"] = sum(group[m] * weights[m] for m in top_models.index)
        results.append(group)
    return pd.concat(results, ignore_index=True)

cv_1w_blended = build_combined_forecast_for_testing(wape_per_product_1w, cv_1w, sparse_model_cols, top_n=2)

# Now measure the blended version's real accuracy
blended_error = (cv_1w_blended["blended_forecast"] - cv_1w_blended["y"]).abs().sum()
blended_wape_combined = blended_error / cv_1w_blended["y"].abs().sum()
print("Blended (top-2) Tier 2 WAPE:", blended_wape_combined)


In [ ]:
chosen_abs_err_1w = errors_1w.apply(
    lambda row: row[f"{best_model_per_product_1w[row['unique_id']]}_abs_err"], axis=1
)
blended_wape_hard = chosen_abs_err_1w.sum() / errors_1w["y"].abs().sum()
print("Hard winner-take-all Tier 2 WAPE:", blended_wape_hard)

print("\nComparison:")
print("Hard selection:", blended_wape_hard)
print("Top-2 blend:   ", blended_wape_combined)


In [ ]:
clean_ids = [uid for uid in intermittent_df_1w["unique_id"].unique() if uid not in low_confidence_ids]

errors_1w_clean = errors_1w[errors_1w["unique_id"].isin(clean_ids)]

chosen_abs_err_clean = errors_1w_clean.apply(
    lambda row: row[f"{best_model_per_product_1w[row['unique_id']]}_abs_err"], axis=1
)
blended_wape_clean = chosen_abs_err_clean.sum() / errors_1w_clean["y"].abs().sum()
print("Tier 2 WAPE WITHOUT the 2 flagged products:", blended_wape_clean)
print("Tier 2 WAPE WITH them (original):", blended_wape_hard)


In [ ]:
individual_wape = wape_per_product_1w.min(axis=1).sort_values(ascending=False)
print(individual_wape)


In [ ]:
unpredictable_threshold = 1.0  # 100% WAPE
tier2_unpredictable = individual_wape[individual_wape > unpredictable_threshold].index.tolist()
print("Tier 2 products that should be flagged low-confidence:", tier2_unpredictable)
print("Count:", len(tier2_unpredictable))


In [ ]:
# ---------------------------------------------------------------
# Build the full, rule-based list of unstable products across BOTH tiers
# ---------------------------------------------------------------
unpredictable_threshold = 1.0  # 100% WAPE

tier1_worst = wape_per_product_2w.min(axis=1)
tier1_unpredictable = tier1_worst[tier1_worst > unpredictable_threshold].index.tolist()

tier2_worst = wape_per_product_1w.min(axis=1)
tier2_unpredictable = tier2_worst[tier2_worst > unpredictable_threshold].index.tolist()

flagged_products = tier1_unpredictable + tier2_unpredictable
print("Total unstable products flagged:", len(flagged_products))
print(flagged_products)

# ---------------------------------------------------------------
# Build the Min-Max buffer recommendation for ALL of them
# ---------------------------------------------------------------
buffer_recommendations = []
for product in flagged_products:
    sales = Y_product[Y_product["unique_id"] == product]["y"]
    min_level = sales.mean()                  # the floor — average demand
    max_level = sales.quantile(0.90)           # the ceiling — 90th percentile, not the raw max
    buffer_recommendations.append({
        "unique_id": product,
        "recommended_min": round(min_level, 2),
        "recommended_max": round(max_level, 2),
    })

buffer_df = pd.DataFrame(buffer_recommendations)
print(buffer_df)


In [ ]:
# ---------------------------------------------------------------
# Build the buffer table for ALL unstable products, with a column
# distinguishing the original 3 extreme ones from the rest
# ---------------------------------------------------------------
original_extreme = {
    "Spirits/Reserve/Singleton/734970",
    "Spirits/Premium/Gordons Flavours/768300",
    "Spirits/Reserve/Glenkinchie/624151",
}

buffer_recommendations = []
for product in flagged_products:
    sales = Y_product[Y_product["unique_id"] == product]["y"]
    min_level = sales.mean()
    max_level = sales.quantile(0.90)
    buffer_recommendations.append({
        "unique_id": product,
        "recommended_min": round(min_level, 2),
        "recommended_max": round(max_level, 2),
        "flag_type": "Extreme (original 3)" if product in original_extreme else "High-risk (newly identified)",
    })

buffer_df = pd.DataFrame(buffer_recommendations)

# Save as a NEW file — does not touch the old one
buffer_df.to_csv("../eabl parsed csv/low_confidence_buffer_recommendations_expanded.csv", index=False)
buffer_df


In [ ]:
original_extreme = {
    "Spirits/Reserve/Singleton/734970",
    "Spirits/Premium/Gordons Flavours/768300",
    "Spirits/Reserve/Glenkinchie/624151",
}

tier1_ids = {
    "Spirits/Mainstream/Kenya Cane/717336",
    "Spirits/Mainstream/Kenya Cane/717383",
    "Spirits/Premium/Gordons Flavours/768493",
    "Spirits/Premium/JW Red Label/763938",
    "Spirits/Reserve/Singleton/734970",
}

flagged_products = [
    # Tier 1 — all 5 confirmed
    "Spirits/Mainstream/Kenya Cane/717336",
    "Spirits/Mainstream/Kenya Cane/717383",
    "Spirits/Premium/Gordons Flavours/768493",
    "Spirits/Premium/JW Red Label/763938",
    "Spirits/Reserve/Singleton/734970",
    # Tier 2 — original 8
    "Spirits/Premium/Gordons Flavours/768300",
    "Spirits/Reserve/Glenkinchie/624151",
    "Spirits/Premium/Gilbeys Berry Bramble/800626",
    "Spirits/Premium/Baileys/753019",
    "Spirits/Premium/Smirnoff Cosmopolitan/800452",
    "Spirits/Reserve/Ciroc /680164",
    "Spirits/Mainstream/Captain Morgan Iced Tea/800646",
    "Spirits/Premium/Smirnoff Vodka Flavours/784527",
    # Confirmed via weekly-vs-cumulative
    "Spirits/Premium/JW Black Label/775335",
]

buffer_recommendations = []
for product in flagged_products:
    sales = Y_product[Y_product["unique_id"] == product]["y"]
    min_level = sales.mean()
    max_level = sales.quantile(0.90)
    buffer_recommendations.append({
        "unique_id": product,
        "tier": "Tier 1" if product in tier1_ids else "Tier 2",
        "recommended_min": round(min_level, 2),
        "recommended_max": round(max_level, 2),
        "flag_type": "Extreme (original 3)" if product in original_extreme else "High-risk (newly identified)",
    })

buffer_df = pd.DataFrame(buffer_recommendations)
print("Total flagged products:", len(buffer_df))
print(buffer_df["tier"].value_counts())
buffer_df.to_csv("../eabl parsed csv/low_confidence_buffer_recommendations_expanded.csv", index=False)
buffer_df


In [ ]:
# 1. Rebuild the function to output a clean, long (tidy) table — one row per product+model
def compare_weekly_vs_cumulative(cv_df, model_cols):
    results = []
    for uid, group in cv_df.groupby("unique_id"):
        actual_total = group["y"].sum()
        for m in model_cols:
            weekly_wape = (group[m] - group["y"]).abs().sum() / group["y"].abs().sum()
            cumulative_error = abs(group[m].sum() - actual_total) / actual_total if actual_total > 0 else None
            results.append({
                "unique_id": uid,
                "model": m,
                "actual_total": round(actual_total, 2),
                "weekly_wape": weekly_wape,
                "weekly_wape_pct": round(weekly_wape * 100, 2) if weekly_wape is not None else None,
                "cumulative_error": cumulative_error,
                "cumulative_error_pct": round(cumulative_error * 100, 2) if cumulative_error is not None else None,
            })
    return pd.DataFrame(results)

# 2. Run it for Tier 2
comparison_1w = compare_weekly_vs_cumulative(cv_1w, sparse_model_cols)

# 3. Save to CSV
comparison_1w.to_csv("../eabl parsed csv/tier2_weekly_vs_cumulative.csv", index=False)
print(comparison_1w)



In [ ]:
comparison_2w = compare_weekly_vs_cumulative(cv_2w, sparse_model_cols)
comparison_2w.to_csv("../eabl parsed csv/tier1_weekly_vs_cumulative.csv", index=False)
print(comparison_2w)


In [ ]:
# Tier 2 overall WAPE — same approach
tier2_errors = []
for uid, model in best_model_per_product_1w.items():
    product_rows = cv_1w[cv_1w["unique_id"] == uid]
    abs_err = (product_rows[model] - product_rows["y"]).abs().sum()
    actual = product_rows["y"].abs().sum()
    tier2_errors.append({"abs_err": abs_err, "actual": actual})

tier2_df = pd.DataFrame(tier2_errors)
tier2_overall_wape = tier2_df["abs_err"].sum() / tier2_df["actual"].sum()
print(f"Tier 2 overall WAPE: {tier2_overall_wape:.2%}")

### Tier 3 — 38 SKUs with too little data to test at all

These are the sparse products with fewer than 14 rows of real history — not enough to run even
one honest cross-validation test window. Instead of testing, these go straight to `.forecast()`
using `CrostonSBA` as a safe, literature-recommended default (Syntetos & Boylan), since there's
no evidence available to pick a genuine "winner" per product. Every row here is flagged
`tested=False`, so it's never mistaken for an evidence-backed number downstream.


In [ ]:
# ---------------------------------------------------------------
# Tier 3 — build the default forecast for the 38 untestable SKUs
# ---------------------------------------------------------------

# Safety rebuild, in case these were lost after a kernel restart
try:
    print("too_short_for_any_cv already exists:", len(too_short_for_any_cv))
except NameError:
    series_lengths_sparse = intermittent_df.groupby("unique_id").size().sort_values()
    too_short_for_any_cv = series_lengths_sparse[series_lengths_sparse < 14]
    print("Rebuilt too_short_for_any_cv:", len(too_short_for_any_cv))

ids_untestable = too_short_for_any_cv.index
intermittent_df_untestable = intermittent_df[intermittent_df["unique_id"].isin(ids_untestable)]

forecast_untestable = sf_intermittent.forecast(df=intermittent_df_untestable, h=13)

final_untestable = forecast_untestable[["unique_id", "ds", "CrostonSBA"]].rename(
    columns={"CrostonSBA": "chosen_model_forecast"}
)
final_untestable["chosen_model_forecast"] = final_untestable["chosen_model_forecast"].clip(lower=0)
final_untestable["tested"] = False

# Confirmation checks
print("\nTier 3 SKUs:", final_untestable["unique_id"].nunique(), "(should be 38)")
print("Total rows:", final_untestable.shape[0], "(should be 494)")
print(final_untestable[["unique_id", "ds", "chosen_model_forecast", "tested"]].head(10))



In [ ]:
print("=== Tier 1 ===")
print("Historical data rows (intermittent_df_2w):", intermittent_df_2w.shape[0])
print("Final forecast rows (final_2w):", final_2w.shape[0], "(should be 27 × 13 = 351)")

print("\n=== Tier 2 ===")
print("Historical data rows (intermittent_df_1w):", intermittent_df_1w.shape[0])
print("Final forecast rows (final_1w):", final_1w.shape[0], "(should be 16 × 13 = 208)")

print("\n=== Tier 3 ===")
print("Historical data rows (intermittent_df_untestable):", intermittent_df_untestable.shape[0])
print("Final forecast rows (final_untestable):", final_untestable.shape[0], "(should be 38 × 13 = 494)")

print("\n=== Total check ===")
total_products = final_2w["unique_id"].nunique() + final_1w["unique_id"].nunique() + final_untestable["unique_id"].nunique()
total_forecast_rows = final_2w.shape[0] + final_1w.shape[0] + final_untestable.shape[0]
print("Total products across all tiers:", total_products, "(should be 81)")
print("Total forecast rows across all tiers:", total_forecast_rows, "(should be 1053)")


In [ ]:
final_forecast_sparse = pd.concat([
    final_2w[["unique_id", "ds", "chosen_model_forecast"]],
    final_1w[["unique_id", "ds", "chosen_model_forecast"]],
    final_untestable[["unique_id", "ds", "chosen_model_forecast"]],
], ignore_index=True)

print("Total unique SKUs:", final_forecast_sparse["unique_id"].nunique(), "(should be 81)")
print("Total rows:", final_forecast_sparse.shape[0], "(should be 1053)")
print(final_forecast_sparse.head(10))


In [ ]:
final_all_skus = pd.concat([
    final_forecast[["unique_id", "ds", "chosen_model_forecast"]],
    final_forecast_sparse,
], ignore_index=True)

print("Total unique SKUs:", final_all_skus["unique_id"].nunique(), "(should be 222)")
print("Total rows:", final_all_skus.shape[0], "(should be 222 × 13 = 2886)")

final_all_skus.to_csv("../eabl parsed csv/final_forecast_all_skus.csv", index=False)
print("Saved.")
